# เทรน LoRA ลายผ้าไหมมัดหมี่อีสาน

โน้ตบุ๊กนี้เทรน LoRA บน Stable Diffusion XL ด้วย dataset ที่เตรียมจาก `preprocess.py`

**ก่อนเริ่ม** ไปที่ Runtime → Change runtime type → เลือก **T4 GPU**

| ขั้น | เวลาโดยประมาณบน T4 |
|---|---|
| ติดตั้ง | 3-5 นาที |
| ขยายภาพด้วย Real-ESRGAN | 2-3 นาที |
| เทรน ~860 steps | 45-70 นาที |
| เจนภาพทดสอบทุก checkpoint | 10-15 นาที |

**ข้อควรรู้เรื่องข้อมูล** ภาพต้นฉบับมาจาก PDF ความละเอียดจริงแค่ 422-745px
tile จึงอยู่ที่ 288px เท่านั้น โน้ตบุ๊กนี้ขยายด้วย Real-ESRGAN เป็น 576px ก่อนเทรน
ซึ่งให้ขอบลายคมกว่าการขยายธรรมดามาก แต่ไม่ได้สร้างรายละเอียดที่ต้นฉบับไม่มี
ถ้าถ่ายผ้าจริงที่ 2000px+ ได้เมื่อไร คุณภาพจะกระโดดกว่าการจูนค่าใด ๆ ในนี้


## 1. ตรวจ GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'ยังไม่ได้เปิด GPU — Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2. ติดตั้ง


In [ ]:
!pip install -q 'diffusers[torch]==0.31.0' transformers==4.46.3 accelerate==1.1.1 \
    peft==0.13.2 bitsandbytes==0.44.1 datasets==3.1.0 safetensors sentencepiece
!git clone -q --depth 1 --branch v0.31.0 https://github.com/huggingface/diffusers.git /content/diffusers_src
print('ติดตั้งเสร็จ')


## 3. อัปโหลด dataset

รัน `python training/make_bundle.py` บนเครื่องก่อน แล้วอัปโหลด `v2_bundle.zip` ที่ได้


In [ ]:
from google.colab import files
import zipfile, pathlib, json

uploaded = files.upload()
name = next(iter(uploaded))

root = pathlib.Path('/content/dataset')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as bundle:
    bundle.extractall(root)

summary = json.loads((root / 'dataset.json').read_text(encoding='utf-8'))
TRIGGER = summary['trigger']
print(summary['tiles'], 'tiles ·', summary['tile_size'], 'px · trigger =', TRIGGER)
print('ตระกูลลาย:', summary['families'])


## 4. ขยายภาพด้วย Real-ESRGAN

tile 288px เล็กเกินกว่าที่ SDXL จะเรียนได้ดี การขยายแบบ bicubic ทำให้ขอบลายฟุ้ง
แล้วโมเดลจะเรียนความเบลอติดไปด้วย Real-ESRGAN รักษาขอบได้ดีกว่ามาก

ถ้าเซลล์นี้พัง ข้ามไปได้ มันจะใช้ภาพขนาดเดิมแทนโดยอัตโนมัติ


In [ ]:
import subprocess, sys, pathlib, shutil
from PIL import Image

SRC = pathlib.Path('/content/dataset/images')
DST = pathlib.Path('/content/dataset_up/images')
DST.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'realesrgan', 'basicsr'], check=True)
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    import cv2

    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    upsampler = RealESRGANer(
        scale=4,
        model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model=model, tile=0, half=True,
    )
    for path in sorted(SRC.glob('*.png')):
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        output, _ = upsampler.enhance(image, outscale=2)
        cv2.imwrite(str(DST / path.name), output)
    print('ขยายด้วย Real-ESRGAN เรียบร้อย')
except Exception as error:
    print('ขยายไม่สำเร็จ ใช้ภาพขนาดเดิมแทน:', str(error)[:200])
    for path in sorted(SRC.glob('*.png')):
        shutil.copy(path, DST / path.name)

shutil.copy(SRC / 'metadata.jsonl', DST / 'metadata.jsonl')
print('ขนาด tile ที่จะเทรน:', Image.open(next(DST.glob('*.png'))).size)


## 5. ตั้งค่าการเทรน

ค่าเหล่านี้ตั้งมาสำหรับ dataset ~143 tiles
ถ้าขยาย dataset เป็น 500+ ให้ลด `EPOCHS` และเพิ่ม `RANK`


In [ ]:
RESOLUTION = 768      # SDXL ชอบ 1024 แต่ข้อมูลเราไม่ถึง 768 คือจุดสมดุล
RANK = 16             # ความจุของ LoRA — 16 พอสำหรับข้อมูลระดับนี้
LR = 1e-4
BATCH = 1             # T4 มี 16GB เท่านั้น
GRAD_ACCUM = 4        # effective batch = 4
EPOCHS = 24

import pathlib, math
tiles = len(list(pathlib.Path('/content/dataset_up/images').glob('*.png')))
steps_per_epoch = math.ceil(tiles / (BATCH * GRAD_ACCUM))
MAX_STEPS = steps_per_epoch * EPOCHS
CHECKPOINT_EVERY = steps_per_epoch * 4

print(tiles, 'tiles ·', steps_per_epoch, 'steps/epoch · รวม', MAX_STEPS, 'steps')
print('เก็บ checkpoint ทุก', CHECKPOINT_EVERY, 'steps → ได้', MAX_STEPS // CHECKPOINT_EVERY, 'จุด')


## 6. เทรน


In [ ]:
!accelerate launch --mixed_precision=fp16 \
  /content/diffusers_src/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --pretrained_vae_model_name_or_path='madebyollin/sdxl-vae-fp16-fix' \
  --train_data_dir='/content/dataset_up/images' \
  --caption_column='text' \
  --resolution={RESOLUTION} --random_flip \
  --train_batch_size={BATCH} \
  --gradient_accumulation_steps={GRAD_ACCUM} \
  --gradient_checkpointing \
  --max_train_steps={MAX_STEPS} \
  --checkpointing_steps={CHECKPOINT_EVERY} \
  --learning_rate={LR} \
  --lr_scheduler='cosine' --lr_warmup_steps=50 \
  --use_8bit_adam \
  --rank={RANK} \
  --seed=1337 \
  --output_dir='/content/lora_out'


**ถ้าเจอ CUDA out of memory** ลดทีละขั้น

1. ตั้ง `RESOLUTION = 640` แล้วรันเซลล์ 5 กับ 6 ใหม่
2. `RESOLUTION = 512`
3. ยังไม่พอ ให้ไปใช้ SD 1.5 ที่เซลล์ท้ายโน้ตบุ๊ก


## 7. เจนภาพทดสอบจากทุก checkpoint

ใช้ prompt ชุดเดิมและ seed ชุดเดิมทุก checkpoint เพื่อให้เทียบกันอย่างยุติธรรม

prompt แบ่งสามกลุ่ม กลุ่มแรกทดสอบว่าเรียนสำเร็จไหม กลุ่มสองทดสอบว่าผสมลายใหม่ได้ไหม
ซึ่งเป็นเป้าหมายหลัก ส่วนกลุ่มสามเป็นลายที่ไม่มีใน dataset ใช้จับ overfitting


In [ ]:
PROMPTS = {
  'A': TRIGGER + ', naga motif with flame crest, burnt orange ground, cream figures',
  'B': TRIGGER + ', tiered offering motif with finial, violet ground, white figures',
  'C': TRIGGER + ', concentric lozenge lattice with floral centre, indigo ground',
  'D': TRIGGER + ', S-hook motif between warp bands, dark maroon ground, cream figures',
  'E': TRIGGER + ', naga motif inside lozenge lattice, indigo ground, three tones of blue',
  'F': TRIGGER + ', tiered offering motif with hook filler, green ground, gold figures',
  'G': TRIGGER + ', spiral and scroll motif, two tone gold and black',
  'H': TRIGGER + ', lotus motif, pink ground, white figures',
  'I': TRIGGER + ', horizontal banded repeat, teal ground',
  'J': TRIGGER,
}
SEEDS = [1337, 2024, 7777, 42242]
print(len(PROMPTS), 'prompts ×', len(SEEDS), 'seeds =', len(PROMPTS) * len(SEEDS), 'ภาพต่อ checkpoint')


In [ ]:
import torch, pathlib, gc
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
).to('cuda')
pipe.set_progress_bar_config(disable=True)

NEGATIVE = 'photograph of a person, text, watermark, blurry, frame, border'
out_root = pathlib.Path('/content/samples')

checkpoints = sorted(
    pathlib.Path('/content/lora_out').glob('checkpoint-*'),
    key=lambda p: int(p.name.split('-')[1]),
)
checkpoints.append(pathlib.Path('/content/lora_out'))

for ckpt in checkpoints:
    if not list(ckpt.glob('*.safetensors')):
        continue
    name = ckpt.name if ckpt.name.startswith('checkpoint') else 'final'
    folder = out_root / name
    folder.mkdir(parents=True, exist_ok=True)

    pipe.load_lora_weights(str(ckpt))
    for key, prompt in PROMPTS.items():
        for seed in SEEDS:
            generator = torch.Generator('cuda').manual_seed(seed)
            image = pipe(
                prompt=prompt, negative_prompt=NEGATIVE,
                num_inference_steps=28, guidance_scale=6.0,
                width=1024, height=1024, generator=generator,
            ).images[0]
            image.save(folder / (key + '_' + str(seed) + '.png'))
    pipe.unload_lora_weights()
    print('เจนเสร็จ', name)
    gc.collect(); torch.cuda.empty_cache()

print('ทั้งหมดอยู่ที่', out_root)


## 8. ดูผลเทียบกัน


In [ ]:
import pathlib
from PIL import Image
import matplotlib.pyplot as plt

KEY = 'E'      # เปลี่ยนเป็น A-J เพื่อดู prompt อื่น
SEED = 1337

folders = sorted(pathlib.Path('/content/samples').iterdir(),
                 key=lambda p: (p.name == 'final', p.name))
found = [(f.name, f / (KEY + '_' + str(SEED) + '.png')) for f in folders]
found = [(n, p) for n, p in found if p.exists()]

fig, axes = plt.subplots(1, len(found), figsize=(4 * len(found), 4.6))
if len(found) == 1:
    axes = [axes]
for ax, (name, path) in zip(axes, found):
    ax.imshow(Image.open(path)); ax.set_title(name, fontsize=10); ax.axis('off')
plt.suptitle('prompt ' + KEY + ' · seed ' + str(SEED))
plt.tight_layout(); plt.show()


## 9. ดาวน์โหลด LoRA กับภาพตัวอย่าง

เอา `samples.zip` ไปให้คะแนนด้วย `evaluate.py` บนเครื่อง
และเอา `lora_weights.zip` ไปใช้เจนจริงต่อ


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/lora_weights', 'zip', '/content/lora_out')
shutil.make_archive('/content/samples', 'zip', '/content/samples')
files.download('/content/lora_weights.zip')
files.download('/content/samples.zip')


---

## ถ้า T4 ไม่ไหว ให้ใช้ SD 1.5 แทน

SD 1.5 เทรนเร็วกว่า SDXL ราว 5 เท่าและกินหน่วยความจำน้อยกว่ามาก
พื้นผิวสู้ SDXL ไม่ได้ แต่ถ้า Colab หลุดบ่อยก็คุ้มกว่า
เอาคำสั่งข้างล่างไปแทนเซลล์ที่ 6 (ลบ # ออกก่อน)


In [ ]:
# !accelerate launch --mixed_precision=fp16 \
#   /content/diffusers_src/examples/text_to_image/train_text_to_image_lora.py \
#   --pretrained_model_name_or_path='runwayml/stable-diffusion-v1-5' \
#   --train_data_dir='/content/dataset_up/images' --caption_column='text' \
#   --resolution=512 --random_flip --train_batch_size=2 \
#   --gradient_accumulation_steps=2 --max_train_steps=2000 \
#   --checkpointing_steps=400 --learning_rate=1e-4 \
#   --lr_scheduler='cosine' --lr_warmup_steps=50 --rank=16 \
#   --seed=1337 --output_dir='/content/lora_out'
